In [ ]:
%%writefile matrix_mul4.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 4

__global__ void matrixMul(int *A, int *B, int *C, int size) {
    int row = threadIdx.y;
    int col = threadIdx.x;

    if (row < size && col < size) {
        int sum = 0;
        for (int k = 0; k < size; k++) {
            sum += A[row * size + k] * B[k * size + col];
        }
        C[row * size + col] = sum;
    }
}

int main() {
    int A[N][N], B[N][N], C[N][N];

    // Initialize matrices
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            A[i][j] = i + j;
            B[i][j] = i * j;
        }
    }

    int *d_A, *d_B, *d_C;
    size_t matrixSize = N * N * sizeof(int);

    cudaMalloc((void**)&d_A, matrixSize);
    cudaMalloc((void**)&d_B, matrixSize);
    cudaMalloc((void**)&d_C, matrixSize);

    cudaMemcpy(d_A, A, matrixSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, matrixSize, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(N, N);

    // Launch kernel
    matrixMul<<<1, threadsPerBlock>>>(d_A, d_B, d_C, N);

    cudaMemcpy(C, d_C, matrixSize, cudaMemcpyDeviceToHost);

    printf("Result Matrix:\n");
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            printf("%d ", C[i][j]);
        }
        printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting matrix_mul.cu


In [4]:
!nvcc matrix_mul.cu -o matrix_mul
!./matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Result Matrix:
0 14 28 42 
0 20 40 60 
0 26 52 78 
0 32 64 96 
